In [8]:
"""
Transfer Learning for Mueller Matrix Tissue Segmentation - FINAL WORKING VERSION
Using only M11 element with pretrained encoders from torchvision

All bugs fixed:
- SSL certificate issues
- Int64 tensor interpolation
- Multiprocessing errors
- Channel mismatch in decoder
- Dynamic input size handling
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import json
import logging
import ssl

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

ssl._create_default_https_context = ssl._create_unverified_context


# ============================================================================
# Configuration
# ============================================================================

class Config:
    DATA_DIR = Path("/Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/testing")
    OUTPUT_DIR = Path("./output_m11_transfer_learning")
    
    ENCODER_NAME = 'resnet34'
    USE_PRETRAINED = True
    
    BATCH_SIZE = 4
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    NUM_WORKERS = 0
    PATIENCE = 15
    
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    RANDOM_SEED = 42
    
    INPUT_SIZE = (512, 512)  # Set to fixed size to handle variable image sizes
    # Use None to keep original sizes, but requires custom collate_fn
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ============================================================================
# U-Net Model
# ============================================================================

class UNetWithPretrainedEncoder(nn.Module):
    """U-Net with pretrained encoder - handles variable input sizes"""
    
    def __init__(self, encoder_name='resnet34', num_classes=2, pretrained=True):
        super().__init__()
        
        self.encoder_name = encoder_name
        self.num_classes = num_classes
        
        logger.info(f"Loading encoder: {encoder_name} (pretrained={pretrained})")
        
        try:
            if encoder_name == 'resnet18':
                weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
                encoder = models.resnet18(weights=weights)
                encoder_channels = [64, 64, 128, 256, 512]
            elif encoder_name == 'resnet34':
                weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
                encoder = models.resnet34(weights=weights)
                encoder_channels = [64, 64, 128, 256, 512]
            elif encoder_name == 'resnet50':
                weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
                encoder = models.resnet50(weights=weights)
                encoder_channels = [64, 256, 512, 1024, 2048]
            elif encoder_name == 'mobilenet_v2':
                weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
                encoder = models.mobilenet_v2(weights=weights)
                encoder_channels = [16, 24, 32, 96, 1280]
            else:
                raise ValueError(f"Unsupported encoder: {encoder_name}")
        
        except Exception as e:
            logger.warning(f"Failed to load pretrained: {e}. Using random init...")
            
            if encoder_name == 'resnet18':
                encoder = models.resnet18(weights=None)
                encoder_channels = [64, 64, 128, 256, 512]
            elif encoder_name == 'resnet34':
                encoder = models.resnet34(weights=None)
                encoder_channels = [64, 64, 128, 256, 512]
            elif encoder_name == 'resnet50':
                encoder = models.resnet50(weights=None)
                encoder_channels = [64, 256, 512, 1024, 2048]
            elif encoder_name == 'mobilenet_v2':
                encoder = models.mobilenet_v2(weights=None)
                encoder_channels = [16, 24, 32, 96, 1280]
        
        if 'resnet' in encoder_name:
            self.encoder0 = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
            self.encoder1 = nn.Sequential(encoder.maxpool, encoder.layer1)
            self.encoder2 = encoder.layer2
            self.encoder3 = encoder.layer3
            self.encoder4 = encoder.layer4
        elif encoder_name == 'mobilenet_v2':
            features = encoder.features
            self.encoder0 = features[0:2]
            self.encoder1 = features[2:4]
            self.encoder2 = features[4:7]
            self.encoder3 = features[7:14]
            self.encoder4 = features[14:]
        
        self.enc_channels = encoder_channels
        
        # Decoder - accounts for concatenated skip connections
        self.decoder4 = self._decoder_block(encoder_channels[4] + encoder_channels[3], encoder_channels[3])
        self.decoder3 = self._decoder_block(encoder_channels[3] + encoder_channels[2], encoder_channels[2])
        self.decoder2 = self._decoder_block(encoder_channels[2] + encoder_channels[1], encoder_channels[1])
        self.decoder1 = self._decoder_block(encoder_channels[1] + encoder_channels[0], encoder_channels[0])
        self.decoder0 = self._decoder_block(encoder_channels[0], 64)
        
        self.final_conv = nn.Conv2d(64, num_classes, 1)
    
    def _decoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc0 = self.encoder0(x)
        enc1 = self.encoder1(enc0)
        enc2 = self.encoder2(enc1)
        enc3 = self.encoder3(enc2)
        enc4 = self.encoder4(enc3)
        
        # Decoder with skip connections
        dec4 = F.interpolate(enc4, size=enc3.shape[2:], mode='bilinear', align_corners=True)
        dec4 = torch.cat([dec4, enc3], dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = F.interpolate(dec4, size=enc2.shape[2:], mode='bilinear', align_corners=True)
        dec3 = torch.cat([dec3, enc2], dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = F.interpolate(dec3, size=enc1.shape[2:], mode='bilinear', align_corners=True)
        dec2 = torch.cat([dec2, enc1], dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = F.interpolate(dec2, size=enc0.shape[2:], mode='bilinear', align_corners=True)
        dec1 = torch.cat([dec1, enc0], dim=1)
        dec1 = self.decoder1(dec1)
        
        dec0 = F.interpolate(dec1, size=x.shape[2:], mode='bilinear', align_corners=True)
        dec0 = self.decoder0(dec0)
        
        out = self.final_conv(dec0)
        
        return out


# ============================================================================
# Data Functions
# ============================================================================

def discover_original_samples(data_dir: Path) -> List[Dict]:
    logger.info(f"Discovering samples in {data_dir}")
    
    all_dirs = [d for d in data_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]
    
    sample_groups = {}
    
    for sample_dir in all_dirs:
        name = sample_dir.name
        base_name = name
        for suffix in ['_original', '_rot90', '_rot180', '_rot270', '_flip_h', '_flip_v']:
            base_name = base_name.replace(suffix, '')
        
        if base_name not in sample_groups:
            sample_groups[base_name] = []
        sample_groups[base_name].append(sample_dir)
    
    original_samples = []
    for base_name, dirs in sample_groups.items():
        original_dir = None
        for d in dirs:
            if '_original' in d.name:
                original_dir = d
                break
        
        if original_dir is None:
            original_dir = dirs[0]
        
        npz_files = [f for f in original_dir.glob("*.npz") if not f.name.startswith('.')]
        if npz_files:
            original_samples.append({
                'sample_name': base_name,
                'sample_dir': original_dir,
                'npz_path': npz_files[0]
            })
    
    logger.info(f"Found {len(original_samples)} unique samples")
    return original_samples


def extract_m11_and_mask(npz_path: Path) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    try:
        with np.load(npz_path, allow_pickle=True) as data:
            m11 = None
            if 'nM11s' in data:
                m11 = np.array(data['nM11s'])
            elif 'M11s' in data:
                m11_raw = np.array(data['M11s'])
                m11_min, m11_max = m11_raw.min(), m11_raw.max()
                if m11_max > m11_min:
                    m11 = (m11_raw - m11_min) / (m11_max - m11_min)
                else:
                    m11 = np.zeros_like(m11_raw, dtype=np.float32)
            elif 'nM' in data:
                nM = np.array(data['nM'])
                if nM.ndim == 4 and nM.shape[-2:] == (4, 4):
                    m11 = nM[:, :, 0, 0]
                elif nM.ndim == 3 and nM.shape[-1] == 16:
                    m11 = nM[:, :, 0]
            
            if m11 is None:
                return None
            
            mask = None
            for key in ['annotation_mask', 'tissue_mask', 'os_mask']:
                if key in data:
                    mask_data = data[key]
                    if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                        mask = np.array(mask_data)
                        break
            
            if mask is None:
                return None
            
            if m11.ndim != 2 or mask.ndim != 2:
                return None
            if m11.shape != mask.shape:
                return None
            
            return m11.astype(np.float32), mask.astype(np.int64)
    
    except Exception as e:
        logger.error(f"Error loading {npz_path.name}: {e}")
        return None


# ============================================================================
# Dataset
# ============================================================================

class M11SegmentationDataset(Dataset):
    def __init__(self, sample_paths: List[Path], input_size: Optional[Tuple[int, int]] = None,
                 augment: bool = False, normalize: bool = True):
        self.sample_paths = sample_paths
        self.input_size = input_size
        self.augment = augment
        self.normalize = normalize
        
        self.samples = []
        logger.info(f"Loading {'and augmenting ' if augment else ''}samples...")
        for path in tqdm(sample_paths):
            data = extract_m11_and_mask(path)
            if data is not None:
                self.samples.append((path, data))
        
        logger.info(f"Loaded {len(self.samples)} valid samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, (m11, mask) = self.samples[idx]
        
        if self.input_size is not None:
            m11 = self._resize(m11, self.input_size, is_mask=False)
            mask = self._resize(mask, self.input_size, is_mask=True)
        
        if self.augment:
            m11, mask = self._augment(m11, mask)
        
        m11_rgb = np.stack([m11, m11, m11], axis=0)
        
        if self.normalize:
            mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
            std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
            m11_rgb = (m11_rgb - mean) / std
        
        image = torch.from_numpy(m11_rgb).float()
        mask = torch.from_numpy(mask).long()
        
        return image, mask
    
    def _resize(self, img, size, is_mask=False):
        img_torch = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0)
        mode = 'nearest' if is_mask else 'bilinear'
        resized = F.interpolate(img_torch, size=size, mode=mode, align_corners=None if is_mask else True)
        resized = resized.squeeze().numpy()
        
        if is_mask:
            resized = resized.astype(np.int64)
        
        return resized
    
    def _augment(self, m11, mask):
        if np.random.rand() > 0.5:
            m11 = np.fliplr(m11).copy()
            mask = np.fliplr(mask).copy()
        
        if np.random.rand() > 0.5:
            m11 = np.flipud(m11).copy()
            mask = np.flipud(mask).copy()
        
        k = np.random.randint(0, 4)
        if k > 0:
            m11 = np.rot90(m11, k).copy()
            mask = np.rot90(mask, k).copy()
        
        if np.random.rand() > 0.5:
            alpha = np.random.uniform(0.8, 1.2)
            beta = np.random.uniform(-0.1, 0.1)
            m11 = np.clip(alpha * m11 + beta, 0, 1)
        
        return m11, mask


# ============================================================================
# Loss Functions
# ============================================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        probs = F.softmax(predictions, dim=1)
        num_classes = predictions.shape[1]
        targets_one_hot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        
        intersection = (probs * targets_one_hot).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))
        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    def __init__(self, weight_ce=0.5, weight_dice=0.5):
        super().__init__()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()
    
    def forward(self, predictions, targets):
        return self.weight_ce * self.ce(predictions, targets) + self.weight_dice * self.dice(predictions, targets)


# ============================================================================
# Training Functions
# ============================================================================

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc="Training")
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        
        if outputs.shape[-2:] != masks.shape[-2:]:
            outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear', align_corners=True)
        
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(dataloader)


def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validation")
        for images, masks in pbar:
            images, masks = images.to(device), masks.to(device)
            
            outputs = model(images)
            
            if outputs.shape[-2:] != masks.shape[-2:]:
                outputs = F.interpolate(outputs, size=masks.shape[-2:], mode='bilinear', align_corners=True)
            
            loss = criterion(outputs, masks)
            preds = torch.argmax(outputs, dim=1)
            
            total_loss += loss.item()
            correct += (preds == masks).sum().item()
            total += masks.numel()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(dataloader), correct / total


# ============================================================================
# Main
# ============================================================================

def main():
    Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    torch.manual_seed(Config.RANDOM_SEED)
    np.random.seed(Config.RANDOM_SEED)
    
    logger.info("="*80)
    logger.info("M11 TRANSFER LEARNING - FINAL VERSION")
    logger.info("="*80)
    logger.info(f"Device: {Config.DEVICE}")
    logger.info(f"Encoder: {Config.ENCODER_NAME} (pretrained={Config.USE_PRETRAINED})")
    
    samples = discover_original_samples(Config.DATA_DIR)
    if len(samples) == 0:
        logger.error("No samples found!")
        return
    
    train_samples, temp = train_test_split(samples, test_size=Config.VAL_RATIO + Config.TEST_RATIO, 
                                           random_state=Config.RANDOM_SEED)
    val_samples, test_samples = train_test_split(temp, test_size=Config.TEST_RATIO / (Config.VAL_RATIO + Config.TEST_RATIO),
                                                 random_state=Config.RANDOM_SEED)
    
    logger.info(f"\nData split: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")
    
    train_dataset = M11SegmentationDataset([s['npz_path'] for s in train_samples], 
                                           input_size=Config.INPUT_SIZE, augment=True)
    val_dataset = M11SegmentationDataset([s['npz_path'] for s in val_samples], 
                                         input_size=Config.INPUT_SIZE, augment=False)
    test_dataset = M11SegmentationDataset([s['npz_path'] for s in test_samples], 
                                          input_size=Config.INPUT_SIZE, augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    all_classes = set()
    for _, mask in train_dataset:
        all_classes.update(mask.unique().numpy())
    num_classes = len(all_classes)
    logger.info(f"\nClasses: {num_classes} ({sorted(all_classes)})")
    
    model = UNetWithPretrainedEncoder(
        encoder_name=Config.ENCODER_NAME,
        num_classes=num_classes,
        pretrained=Config.USE_PRETRAINED
    ).to(Config.DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    logger.info(f"Model parameters: {total_params:,}")
    
    criterion = CombinedLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    logger.info("\nStarting training...")
    for epoch in range(Config.NUM_EPOCHS):
        logger.info(f"\nEpoch {epoch+1}/{Config.NUM_EPOCHS}")
        
        train_loss = train_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, Config.DEVICE)
        
        scheduler.step(val_loss)
        
        logger.info(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_acc': val_acc,
            }, Config.OUTPUT_DIR / 'best_model.pth')
            logger.info("✓ Saved best model")
        else:
            patience_counter += 1
        
        if patience_counter >= Config.PATIENCE:
            logger.info(f"Early stopping at epoch {epoch+1}")
            break
    
    with open(Config.OUTPUT_DIR / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    logger.info("\nEvaluating on test set...")
    checkpoint = torch.load(Config.OUTPUT_DIR / 'best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    test_loss, test_acc = validate_epoch(model, test_loader, criterion, Config.DEVICE)
    logger.info(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
    
    with open(Config.OUTPUT_DIR / 'test_results.json', 'w') as f:
        json.dump({'test_loss': test_loss, 'test_acc': test_acc}, f, indent=2)
    
    logger.info("\n" + "="*80)
    logger.info("TRAINING COMPLETE!")
    logger.info("="*80)


if __name__ == "__main__":
    main()

INFO:__main__:================================================================================
INFO:__main__:M11 TRANSFER LEARNING - FINAL VERSION
INFO:__main__:================================================================================
INFO:__main__:Device: cpu
INFO:__main__:Encoder: resnet34 (pretrained=True)
INFO:__main__:Discovering samples in /Volumes/ep_ssd/MPL_Data/mmNoTissueFilter/testing
INFO:__main__:Found 72 unique samples
INFO:__main__:
Data split: Train=50, Val=11, Test=11
INFO:__main__:Loading and augmenting samples...
100%|██████████| 50/50 [00:01<00:00, 29.00it/s]
INFO:__main__:Loaded 49 valid samples
INFO:__main__:Loading samples...
100%|██████████| 11/11 [00:00<00:00, 52.72it/s]
INFO:__main__:Loaded 11 valid samples
INFO:__main__:Loading samples...
100%|██████████| 11/11 [00:00<00:00, 33.03it/s]
INFO:__main__:Loaded 10 valid samples
INFO:__main__:
Classes: 2 ([np.int64(0), np.int64(1)])
INFO:__main__:Loading encoder: resnet34 (pretrained=True)
INFO:__main__:Model